In [1]:
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
df = pd.read_parquet("../data/processed/arxiv_ml.parquet")

df.head()

,id,title,authors,category,text_raw,text_ml
0,704.0001,Calculation of prompt diphoton production cros...,"C. Bal\'azs, E. L. Berger, P. M. Nadolsky, C.-...",hep-ph,Calculation of prompt diphoton production cros...,calculation prompt diphoton production cross s...
1,704.0002,Sparsity-certifying Graph Decompositions,Ileana Streinu and Louis Theran,math.CO,Sparsity-certifying Graph Decompositions We ...,sparsity certify graph decomposition describe ...
2,704.0003,The evolution of the Earth-Moon system based o...,Hongjun Pan,physics.gen-ph,The evolution of the Earth-Moon system based o...,evolution earth moon system base dark matter f...
3,704.0004,A determinant of Stirling cycle numbers counts...,David Callan,math.CO,A determinant of Stirling cycle numbers counts...,determinant stirling cycle number count unlabe...
4,704.0005,From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\a...,Wael Abu-Shammala and Alberto Torchinsky,math.CA,From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\a...,dyadic lambda alpha lambda alpha compute lambd...


In [2]:
model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\DELL\Desktop\Cdac\aiml-project\myenv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\DELL\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
embeddings = model.encode(
    df["text_raw"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

In [6]:
print(embeddings.shape)

(1000, 384)


In [7]:
embeddings[0]

array([-1.58419624e-01, -1.01032890e-02,  2.19406607e-03,  4.50048447e-02,
        3.63339409e-02, -4.24838215e-02, -4.61273342e-02,  5.97007573e-02,
        4.23808470e-02, -3.25520933e-02,  1.74576491e-02, -3.14672515e-02,
       -3.97230797e-02, -3.76211759e-03,  1.54057713e-02,  2.37496514e-02,
       -2.08473932e-02, -5.65718412e-02, -8.76072198e-02, -1.72332283e-02,
       -5.22109009e-02,  2.45143455e-02, -3.29107568e-02,  2.20729280e-02,
        7.40172789e-02, -9.27654058e-02, -3.29313502e-02, -9.33981035e-03,
        4.80712540e-02,  1.16254399e-02,  2.55359132e-02, -1.39606092e-02,
       -2.30887458e-02,  3.90545130e-02,  7.85364956e-02,  4.41619456e-02,
        3.04403864e-02, -9.58790928e-02,  3.70898806e-02, -2.86279954e-02,
        4.57841083e-02,  8.33052211e-03,  2.55478336e-03, -1.64655522e-02,
        6.29489794e-02,  2.37136800e-02,  2.86393948e-02, -5.63222356e-02,
       -5.08890785e-02,  7.98176229e-03,  7.28131458e-02,  5.07254414e-02,
       -1.26286773e-02,  

In [8]:
print(embeddings[0][:10])

[-0.15841962 -0.01010329  0.00219407  0.04500484  0.03633394 -0.04248382
 -0.04612733  0.05970076  0.04238085 -0.03255209]


In [9]:
print(type(embeddings))
print(embeddings.dtype)

<class 'numpy.ndarray'>
float32


In [10]:
semantic_similarity = cosine_similarity(
    embeddings
)

print(semantic_similarity.shape)

(1000, 1000)


In [11]:
indices = pd.Series(
    df.index,
    index=df["title"]
).drop_duplicates()

In [12]:
def semantic_recommend(title, top_n=5):

    idx = indices[title]

    sim_scores = list(
        enumerate(
            semantic_similarity[idx]
        )
    )

    sim_scores = sorted(
        sim_scores,
        key=lambda x: x[1],
        reverse=True
    )

    sim_scores = sim_scores[1:top_n+1]

    paper_indices = [i[0] for i in sim_scores]

    recommendations = df.iloc[
        paper_indices
    ][
        [
            "title",
            "authors",
            "category"
        ]
    ].copy()

    recommendations["Similarity"] = [
        round(score[1] * 100,2)
        for score in sim_scores
    ]

    return recommendations

In [13]:
semantic_recommend(
    df.iloc[0]["title"]
)

,title,authors,category,Similarity
839,Associated production of the charged Higgs bos...,"Yao-Bei Liu, Jie-Fen Shen",hep-ph,64.519997
759,"Search for Heavy, Long-Lived Particles that De...",A. Abulencia et al. (CDF Collaboration),hep-ex,63.930000
641,Direct photons and dileptons via color dipoles,"B. Z. Kopeliovich, A. H. Rezaeian, H. J. Pirne...",hep-ph,60.990002
234,The Determination of the Helicity of $W'$ Boso...,Thomas G. Rizzo,hep-ph,59.459999
637,Polarizations of J/psi and psi(2S) Mesons Prod...,CDF Collaboration,hep-ex,59.310001


In [14]:
semantic_recommend(df.iloc[0]["title"])

,title,authors,category,Similarity
839,Associated production of the charged Higgs bos...,"Yao-Bei Liu, Jie-Fen Shen",hep-ph,64.519997
759,"Search for Heavy, Long-Lived Particles that De...",A. Abulencia et al. (CDF Collaboration),hep-ex,63.930000
641,Direct photons and dileptons via color dipoles,"B. Z. Kopeliovich, A. H. Rezaeian, H. J. Pirne...",hep-ph,60.990002
234,The Determination of the Helicity of $W'$ Boso...,Thomas G. Rizzo,hep-ph,59.459999
637,Polarizations of J/psi and psi(2S) Mesons Prod...,CDF Collaboration,hep-ex,59.310001
